# Feature Extraction

Extract image descriptors or other features for model training.

# Color Histogram Feature Extraction
**Dataset:** CUB-200-2011, first 20 bird species

## Objectives

1. Load the prepared 20-species dataset.
2. Preserve the 70% training, 15% validation and 15% testing split.
3. Implement RGB color-histogram feature extraction.
4. Visualize and review extracted histograms.
5. Extract and save color-histogram feature vectors.
6. Optionally implement Local Binary Patterns (LBP).

In [1]:
# Install the dataset downloader
!pip -q install kagglehub

from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Download the CUB-200-2011 dataset
download_path = Path(
    kagglehub.dataset_download("wenewone/cub2002011")
)

DATASET_ROOT = download_path / "CUB_200_2011"
IMAGES_FOLDER = DATASET_ROOT / "images"

print("Downloaded path:", download_path)
print("Dataset root exists:", DATASET_ROOT.exists())
print("Images folder exists:", IMAGES_FOLDER.exists())

100%|██████████| 1.49G/1.49G [00:18<00:00, 87.2MB/s]

Extracting files...


Downloaded path: /root/.cache/kagglehub/datasets/wenewone/cub2002011/versions/7
Dataset root exists: True
Images folder exists: True


In [3]:
# Load official class information
classes = pd.read_csv(
    DATASET_ROOT / "classes.txt",
    sep=r"\s+",
    names=["class_id", "class_name"]
)

# Load image IDs and relative paths
images = pd.read_csv(
    DATASET_ROOT / "images.txt",
    sep=r"\s+",
    names=["image_id", "image_path"]
)

# Load the class assigned to each image
image_labels = pd.read_csv(
    DATASET_ROOT / "image_class_labels.txt",
    sep=r"\s+",
    names=["image_id", "class_id"]
)

# Combine all metadata
metadata = images.merge(
    image_labels,
    on="image_id",
    how="inner"
)

metadata = metadata.merge(
    classes,
    on="class_id",
    how="inner"
)

print("All dataset images:", len(metadata))
print("All dataset classes:", metadata["class_id"].nunique())

display(metadata.head())

All dataset images: 11788
All dataset classes: 200


,image_id,image_path,class_id,class_name
0,1,001.Black_footed_Albatross/Black_Footed_Albatr...,1,001.Black_footed_Albatross
1,2,001.Black_footed_Albatross/Black_Footed_Albatr...,1,001.Black_footed_Albatross
2,3,001.Black_footed_Albatross/Black_Footed_Albatr...,1,001.Black_footed_Albatross
3,4,001.Black_footed_Albatross/Black_Footed_Albatr...,1,001.Black_footed_Albatross
4,5,001.Black_footed_Albatross/Black_Footed_Albatr...,1,001.Black_footed_Albatross


In [4]:
# Select class IDs 1 to 20, matching the team preparation notebook
selected_class_ids = (
    classes
    .sort_values("class_id")
    .head(20)["class_id"]
    .tolist()
)

metadata_20 = metadata[
    metadata["class_id"].isin(selected_class_ids)
].copy()

# Add the complete image path
metadata_20["full_image_path"] = metadata_20["image_path"].apply(
    lambda relative_path: IMAGES_FOLDER / relative_path
)

metadata_20 = (
    metadata_20
    .sort_values("image_id")
    .reset_index(drop=True)
)

existing_files = metadata_20["full_image_path"].apply(
    lambda image_path: image_path.exists()
).sum()

print("Selected classes:", metadata_20["class_id"].nunique())
print("Selected images:", len(metadata_20))
print("Existing image files:", existing_files)

print("\nSelected bird species:")
display(
    metadata_20[
        ["class_id", "class_name"]
    ].drop_duplicates()
)

Selected classes: 20
Selected images: 1115
Existing image files: 1115

Selected bird species:


,class_id,class_name
0,1,001.Black_footed_Albatross
60,2,002.Laysan_Albatross
120,3,003.Sooty_Albatross
178,4,004.Groove_billed_Ani
238,5,005.Crested_Auklet
282,6,006.Least_Auklet
323,7,007.Parakeet_Auklet
376,8,008.Rhinoceros_Auklet
424,9,009.Brewer_Blackbird
483,10,010.Red_winged_Blackbird


In [5]:
# First split: 70% training and 30% remaining
train_data, remaining_data = train_test_split(
    metadata_20,
    test_size=0.30,
    stratify=metadata_20["class_id"],
    random_state=42
)

# Second split: divide the remaining 30% equally
# into 15% validation and 15% testing
validation_data, test_data = train_test_split(
    remaining_data,
    test_size=0.50,
    stratify=remaining_data["class_id"],
    random_state=42
)

# Add split labels
metadata_20["split"] = ""

metadata_20.loc[train_data.index, "split"] = "train"
metadata_20.loc[validation_data.index, "split"] = "val"
metadata_20.loc[test_data.index, "split"] = "test"

split_counts = metadata_20["split"].value_counts()

print("Dataset split:")
print("Training images:", split_counts["train"])
print("Validation images:", split_counts["val"])
print("Testing images:", split_counts["test"])
print("Total images:", split_counts.sum())

# Confirm that every image received a split
assert (metadata_20["split"] != "").all()

print("\nEvery image received a split successfully.")

Dataset split:
Training images: 780
Validation images: 167
Testing images: 168
Total images: 1115

Every image received a split successfully.
